<a href="https://colab.research.google.com/github/Abrarswahab/interactive-educational-application/blob/main/Model_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. تثبيت المكتبات اللازمة

In [ ]:
!pip install roboflow ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.9/175.9 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 117.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 131.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 158.0 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


### 2. كود التحميل والتدريب (Fine-tuning Script)
سنقوم هنا بتحميل الموديل "الخام" وتدريبه على بياناتك مع ضبط الـ `Hyperparameters` لتحسين النتائج:

In [ ]:
import os
from roboflow import Roboflow
from ultralytics import YOLO

# --- المرحلة الأولى: تحميل الداتا سيت ---
rf = Roboflow(api_key="sRjpLSexmChMgmxvTWnb")
project = rf.workspace("abdulrahman-k-b").project("visual-teacher-for-kids-v3-1")
version = project.version(2)
dataset = version.download("yolov11") # غيرنا الصيغة إلى yolov11 لتناسب الموديل مباشرة

# --- المرحلة الثانية: إعداد الموديل و الـ Fine-tuning ---
# سنستخدم النسخة الـ Small (yolo11s-seg) لأنها تعطي توازن ممتاز بين السرعة والدقة
model = YOLO('yolo11s-seg.pt')

# --- المرحلة الثالثة: بدء التدريب ---
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,           # عدد الدورات، يمكنك زيادته إذا لم تصل للدقة المطلوبة
    imgsz=640,            # الحجم الذي اخترناه في Preprocessing
    batch=16,             # تعتمد على قوة الـ GPU لديك
    patience=20,          # يتوقف التدريب إذا لم يتحسن الموديل لـ 20 دورة (Early Stopping)
    device=0,             # استخدام الـ GPU (استخدم 'cpu' إذا لم يتوفر)

    # إعدادات الـ Fine-tuning لتحسين النتائج:
    lr0=0.01,             # معدل التعلم الابتدائي
    lrf=0.01,             # معدل التعلم النهائي (Cosine annealing)
    dropout=0.1,          # لمنع الـ Overfitting (حفظ الصور بدلاً من فهمها)
    overlap_mask=True,    # ضروري جداً لـ "نطوق" للتعامل مع الأشياء المتداخلة
    mask_ratio=1          # جودة الـ Masks
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to visual-teacher-for-kids-v3.1-2 in yolov11:: 100%|██████████| 30678/30678 [00:01<00:00, 17270.07it/s]


Ultralytics 8.4.38 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/visual-teacher-for-kids-v3.1-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=1, max_det=300, mixup=0.0, mode=train, model=yolo11s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimi

### شرح الإعدادات (Fine-tuning Logic):
* **`yolo11s-seg.pt`**: اخترت لك نسخة **Small** بدلاً من Nano لأن الـ Segmentation يحتاج لقوة أكبر قليلاً لرسم الأقناع بدقة حول الألعاب والفواكه.
* **`patience=20`**: ميزة رائعة، إذا وجد الموديل أنه لم يعد يتعلم شيئاً جديداً سيوقف التدريب تلقائياً ليوفر عليك الوقت.
* **`imgsz=640`**: يجب أن يطابق تماماً ما اخترته في Roboflow للحصول على أفضل النتائج.
* **`dropout=0.1`**: بما أن لديك بعض الكلاسات ببيانات قليلة، هذا الخيار يمنع الموديل من "حفظ" الصور ويجبره على "فهم" الأشكال.

### ملاحظة أمنية هامة:
لقد قمت بمشاركة الـ **API Key** الخاص بك في الكود أعلاه. أنصحك بشدة بعد انتهاء هذا المشروع أن تقوم بتغيير الـ API Key من إعدادات حسابك في Roboflow لضمان خصوصية بياناتك.

**هل تريد مني كوداً إضافياً لتجربة الموديل (Inference) على صورة من جهازك بعد انتهاء التدريب؟**